In [ ]:
import pandas as pd
import numpy as np
import optuna
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Configurazione del logging di Optuna per essere "chiacchierone" (Verbose)
import logging
optuna.logging.set_verbosity(optuna.logging.INFO)

# 1. CARICAMENTO E PREPARAZIONE DATI
data = df_zscore.copy()

X = df_zscore.drop(columns=["price"])
y = df_zscore["price"]
# Usiamo uno split 80/20. Il test set rimane "sacro" per la valutazione finale.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. DEFINIZIONE DELLA FUNZIONE OBIETTIVO
def objective(trial):
    # Spazio di ricerca definito con distribuzioni suggerite
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        # Usiamo scale logaritmiche per parametri che variano su diversi ordini di grandezza
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'eval_metric': 'logloss',
        'random_state': 42
    }

    # Inizializzazione del modello con i parametri del "tentativo" corrente
    model = xgb.XGBClassifier(**param)
    
    # Cross-Validation a 5 fold sul training set
    # L'obiettivo di Optuna è massimizzare la media di questi 5 punteggi
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy').mean()
    
    return score

# 3. CREAZIONE DELLO STUDIO E OTTIMIZZAZIONE
print("\n[INFO] Avvio Ottimizzazione Bayesiana...")
# 'maximize' perché vogliamo l'accuratezza più alta
study = optuna.create_study(direction='maximize')

# Eseguiamo 50 tentativi (Trial)
study.optimize(objective, n_trials=50)

# 4. ANALISI DEI RISULTATI (Correzione Errore best_params_)
print("\n" + "="*30)
print(" RISULTATI OTTIMIZZAZIONE ")
print("="*30)
print(f"Miglior accuratezza trovata in CV: {study.best_value:.4f}")
print("\nMigliori parametri identificati:")

# NOTA: In Optuna si usa .best_params (senza underscore finale)
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

# 5. ADDESTRAMENTO DEL MODELLO "CAMPIONE"
# Ricostruiamo il modello usando la scompattazione del dizionario dei parametri migliori
best_model = xgb.XGBClassifier(**study.best_params, eval_metric='logloss', random_state=42)
best_model.fit(X_train, y_train)

# 6. VALUTAZIONE FINALE SUL TEST SET (Dati mai visti)
y_pred = best_model.predict(X_test)

# Visualizzazione della Matrice di Confusione
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test, 
    display_labels=data.target_names, 
    cmap='GnBu', ax=ax
)
plt.title("Matrice di Confusione Finale (Best Optuna Model)")
plt.grid(False) # Rimuove le linee della griglia per chiarezza
plt.show()

# 7. REPORT FINALE DI PERFORMANCE
print("\n" + "="*30)
print(" REPORT TECNICO FINALE ")
print("="*30)
print(classification_report(y_test, y_pred, target_names=data.target_names))